In [36]:
# manejo de datos
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# api Xenocanto
from xenopy import Query
from scipy import signal
import scipy.signal.windows

# manejo de audio
import librosa
import librosa.display

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [37]:
# Ruta al archivo MP3
ruta_al_archivo = 'prueba.mp3'

# Cargar el archivo MP3
y, sr = librosa.load(ruta_al_archivo)

# 'y' es un array de numpy que contiene los datos de audio
# 'sr' es la frecuencia de muestreo del audio (número de muestras por segundo)

In [38]:
librosa.load(ruta_al_archivo)

(array([-2.7939677e-09, -5.5879354e-09, -4.6566129e-09, ...,
         6.9249336e-07, -1.9484100e-06, -5.0419339e-06], dtype=float32),
 22050)

In [3]:
# Set the hop length; at 22050 Hz, 512 samples ~= 23ms
hop_length = 512

# Separate harmonics and percussives into two waveforms
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [4]:
# Beat track on the percussive signal
tempo, beat_frames = librosa.beat.beat_track(y=y_percussive, sr=sr)

In [5]:
# Compute MFCC features from the raw signal
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)

In [6]:
# And the first-order differences (delta features)
mfcc_delta = librosa.feature.delta(mfcc)

In [7]:
# Stack and synchronize between beat events
# This time, we'll use the mean value (default) instead of median
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [8]:
# Compute chroma features from the harmonic signal
chromagram = librosa.feature.chroma_cqt(y=y_harmonic, sr=sr)

In [9]:
# Extraer características de audio
features = {
    'Longitud de onda': len(y), # Es la longitud total de la señal de audio.
    'Intensidad miníma': np.min(librosa.feature.rms(y=y)), # Es el valor mínimo de la raíz cuadrada media (RMS) de la señal de audio.
    'Intensidad media': np.mean(librosa.feature.rms(y=y)), # Es el valor medio de la RMS de la señal de audio.
    'Intensidad máxima': np.max(librosa.feature.rms(y=y)), # Es el valor máximo de la RMS de la señal de audio.
    'Tonos principales': len(librosa.core.piptrack(y=y)[1]), # Es el número de tonos principales en la señal de audio.
    'Melodía': np.mean(librosa.feature.tonnetz(y=y, sr=sr)), # Es la melodía promedio de la señal de audio.
    'Centroide espectral': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)), # Es el centroide espectral promedio de la señal de audio.
    'Rolloff espectral': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)), # Es la frecuencia por debajo de la cual se encuentra una cierta cantidad de la energía espectral.
    'Contraste espectral': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)), # Es la diferencia entre los picos y los valles en un espectro.
    'Ancho de banda espectral': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)), # Es el ancho de la banda de frecuencias en la que se encuentra la mayor parte de la energía.
    'Croma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)), # Es una representación de la distribución de energía por tonos.
    'Tempo': librosa.beat.tempo(y=y, sr=sr)[0], # Es la velocidad o el ritmo de una pieza musical.
    'Pulso': np.mean(librosa.feature.tempogram(y=y, sr=sr)), # Es una representación del ritmo en una señal de audio.
    'Coeficientes MFCC': np.mean(librosa.feature.mfcc(y=y, sr=sr)), # Son los coeficientes de los cepstrum de frecuencia mel, que son una representación del espectro de potencia de una señal de audio
    'RMS': np.mean(librosa.feature.rms(y=y)), # Es la raíz cuadrada media de la señal de audio.
    'Cens': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)), # Es una versión suavizada del croma.
    'Piptrack': np.mean(librosa.core.piptrack(y=y)[1]), # Es una representación de los picos de una señal de audio.
    'Cruces por cero': np.mean(librosa.feature.zero_crossing_rate(y)), # Es la tasa a la que la señal cambia de signo.
    'Cromagrama CQT': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)), # Es una representación del croma utilizando una transformada de calidad constante.
    'Cromagrama CENS': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)), # Es una versión suavizada del cromagrama CQT.
    'Melspectrogram': np.mean(librosa.feature.melspectrogram(y=y, sr=sr)), # Es una representación del espectro de potencia en una escala de frecuencia mel.
    'Polynomial coefficients': np.mean(librosa.feature.poly_features(y=y, sr=sr)), # Son los coeficientes de un polinomio que se ajusta a la señal de audio.
    'Fourier tempogram': np.mean(librosa.feature.fourier_tempogram(y=y, sr=sr)), # Es una representación del ritmo basada en la transformada de Fourier.
    'Ceros Cruzados': np.mean(librosa.feature.zero_crossing_rate(y)), # Es la tasa a la que la señal cambia de signo.
    'Perceptrual Sharpness': np.mean(librosa.feature.spectral_flatness(y=y)), # Es una medida de la nitidez percibida de la señal de audio.
    'Loudness': np.sum(librosa.feature.rms(y=y)), # Es la suma de la RMS de la señal de audio.
    'Tonnetz': np.mean(librosa.feature.tonnetz(y=y, sr=sr)), # Es una representación de la tonalidad de una señal de audio.
    'Contraste': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)), # Es la diferencia entre los picos y los valles en un espectro.
    'Rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)), # Es la frecuencia por debajo de la cual se encuentra una cierta cantidad de la energía espectral.
    'Centroide': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)), # Es el centro de gravedad del espectro.
    'Bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)) # Es el ancho de la banda de frecuencias en la que se encuentra la mayor parte de la energía.
    } 

# Crear un DataFrame para almacenar los datos
df = pd.DataFrame(features, index=[0])

# Guardar los datos en un archivo CSV
#df.to_csv('datos_de_audio.csv', index=False)
df.head()

,Longitud de onda,Intensidad miníma,Intensidad media,Intensidad máxima,Tonos principales,Melodía,Centroide espectral,Rolloff espectral,Contraste espectral,Ancho de banda espectral,Croma,Tempo,Pulso,Coeficientes MFCC,RMS,Cens,Piptrack,Cruces por cero,Cromagrama CQT,Cromagrama CENS,Melspectrogram,Polynomial coefficients,Fourier tempogram,Ceros Cruzados,Perceptrual Sharpness,Loudness,Tonnetz,Contraste,Rolloff,Centroide,Bandwidth
0,197568,0.032679,0.081596,0.194569,1025,0.000042,5639.317557,7371.802833,21.260144,1898.440747,0.337589,129.199219,0.250223,-24.013008,0.081596,0.282174,0.055431,0.541826,0.622486,0.282174,0.356371,0.556976,0.427478-0.009859j,0.541826,0.039559,31.495869,0.000042,21.260144,7371.802833,5639.317557,1898.440747


In [10]:
# Aggregate chroma features between beat events
# We'll use the median value of each feature between beat frames
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [11]:
# Finally, stack all beat-synchronous features together
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])

In [12]:
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [13]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)
mfcc

array([[-3.6174402e+02, -2.0442473e+02, -1.5977348e+02, ...,
        -2.2161380e+02, -2.1371092e+02, -2.3938112e+02],
       [-5.1079769e+01, -6.0956253e+01, -7.3678589e+01, ...,
        -5.6710976e+01, -4.8849056e+01, -4.4990215e+01],
       [-8.0093445e+01, -8.5126793e+01, -9.6285263e+01, ...,
        -4.4385803e+01, -3.5310326e+01, -3.2996597e+01],
       ...,
       [ 2.8529568e+00, -4.1075377e+00, -4.5974665e+00, ...,
        -6.5560570e+00, -3.2641020e+00, -5.5949205e-01],
       [-5.6567103e-01,  2.0946629e+00, -1.8572937e+00, ...,
        -5.7987604e+00,  4.8781931e-02,  4.6487370e+00],
       [-1.9916754e+00, -3.8413596e+00, -9.5599451e+00, ...,
         2.5831614e+00,  6.3585157e+00,  6.2258711e+00]], dtype=float32)

In [14]:
mfcc_delta = librosa.feature.delta(mfcc)
mfcc_delta

array([[16.855936  , 16.855936  , 16.855936  , ..., -0.89970374,
        -0.89970374, -0.89970374],
       [-3.4494283 , -3.4494283 , -3.4494283 , ...,  1.7929302 ,
         1.7929302 ,  1.7929302 ],
       [-4.0492163 , -4.0492163 , -4.0492163 , ...,  1.7142378 ,
         1.7142378 ,  1.7142378 ],
       ...,
       [-3.6456344 , -3.6456344 , -3.6456344 , ...,  1.7010834 ,
         1.7010834 ,  1.7010834 ],
       [-1.1730183 , -1.1730183 , -1.1730183 , ...,  1.0955445 ,
         1.0955445 ,  1.0955445 ],
       [ 1.2888954 ,  1.2888954 ,  1.2888954 , ...,  0.47902873,
         0.47902873,  0.47902873]], dtype=float32)

In [20]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [16]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [17]:
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [21]:
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])

array([[ 4.67628032e-01,  3.81025225e-01,  5.37832201e-01,
         7.70256639e-01,  4.52672452e-01,  6.34892344e-01,
         6.80325508e-01,  6.66120291e-01,  6.86720252e-01,
         5.92064261e-01],
       [ 6.63884103e-01,  8.00696492e-01,  6.51479423e-01,
         5.44066787e-01,  4.15778190e-01,  4.71823215e-01,
         5.26662886e-01,  4.97034311e-01,  5.95123410e-01,
         4.78417754e-01],
       [ 6.79648817e-01,  7.55570889e-01,  7.84529269e-01,
         6.67085826e-01,  5.80109656e-01,  6.16851926e-01,
         5.05798936e-01,  6.28109992e-01,  7.35601366e-01,
         6.36415601e-01],
       [ 6.55618548e-01,  5.51733077e-01,  7.54269540e-01,
         6.07961774e-01,  7.84373701e-01,  6.68724597e-01,
         5.65968454e-01,  6.02748096e-01,  6.85563564e-01,
         6.60624802e-01],
       [ 8.75716031e-01,  5.65337181e-01,  6.06589913e-01,
         5.02276063e-01,  4.20100778e-01,  5.15054107e-01,
         4.78921622e-01,  5.38660228e-01,  6.13360345e-01,
         5.